## Transformación de la data: KDD con Apache Spark

Se hacen las transformaciones debidas para trabajar posteriormente con la data

> Continúa el informe del EDA: aquí se limpia la data, se crean las columnas que necesita cada técnica y se deja
> el dataset final listo en `parquet/procesos/`.

#### Imports

Importa las librerías de Spark y pandas que usan las celdas de selección, limpieza y transformación.

In [6]:
# Utilidades del sistema y manejo de rutas
import os, socket
from pathlib import Path

# pandas solo recibe tablas pequeñas ya agregadas por Spark
import pandas as pd

# Spark: sesión, ventanas y funciones de columnas
from pyspark.sql import SparkSession, Window, functions as F

# Tokenizador y stopwords en español para preparar el texto
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

pd.set_option("display.max_colwidth", 80)

#### Utilidades

Calcula la ruta del proyecto y crea la sesión de Spark.

- Constantes

In [7]:
# Si el notebook se abre desde 01_EDA_KDD/, subimos un nivel para encontrar la carpeta parquet/
proyecto = Path.cwd()
if not (proyecto / "parquet").exists() and (proyecto.parent / "parquet").exists():
    proyecto = proyecto.parent     # el notebook vive en 01_EDA_KDD/, el repo está un nivel arriba

parquets = proyecto / "parquet"
anios = [2023, 2024, 2025]

- Spark

In [ ]:
# Si estamos dentro de Docker se usa el cluster (spark://...); si no, Spark local con todos los núcleos
spark_master = os.environ.get("spark_master", "local[*]")

builder = (SparkSession.builder
           .appName("kdd")
           .master(spark_master)
           .config("spark.driver.memory",   os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
           .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "4g"))
           .config("spark.sql.shuffle.partitions", "24")        # suficiente para ~230 mil filas
           .config("spark.ui.showConsoleProgress", "false")     # evita la barra de progreso en las salidas
           .config("spark.sql.session.timeZone", "America/Lima"))

# En modo cluster, los executors necesitan saber cómo volver a contactar al driver (este notebook)
if spark_master.startswith("spark://"):
    builder = (builder.config("spark.driver.host", socket.gethostname())
                      .config("spark.driver.bindAddress", "0.0.0.0"))

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")   # solo mostramos errores, no avisos

### 1. Verificación del cluster

Confirma que el notebook está conectado al cluster de Spark del contenedor

In [9]:
# Un vistazo rápido a los recursos con los que vamos a trabajar
sc = spark.sparkContext
estado = sc._jsc.sc().getExecutorMemoryStatus()

print(f"Master        : {sc.master}")
print(f"Executors     : {estado.size() - 1}")     # se resta 1 porque el driver también aparece en la lista
print(f"Cores totales : {sc.defaultParallelism}")
print(f"Memoria driver: {sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3:.1f} GB")
print("UI del job    : http://localhost:4040")

Master        : spark://spark-master:7077
Executors     : 1
Cores totales : 2
Memoria driver: 3.0 GB
UI del job    : http://localhost:4040


### 2. Cargar dataset

Lee la data cargada en `a_eda.ipynb`

In [10]:
# Leemos la vista plana que dejó el EDA en parquet/staging
base = spark.read.parquet(str(parquets / "staging")).cache()
N = base.count()

### 3. Áreas de análisis

**P1 — Competencia.** ¿Qué combinaciones de método, monto, rubro y región se asocian a procesos con
**un solo postor**? (baja competencia = mayor riesgo de sobrecosto)

**P2 — Fraccionamiento / procesos repetidos.** ¿Hay procesos **casi idénticos** de una misma entidad, en fechas
cercanas y con montos bajos, que sugieran dividir una compra para evadir un método más exigente?

**P3 — Precios de referencia.** Para procesos con **objeto parecido**, ¿cuánto varía el monto contratado entre
entidades y regiones?

#### 3.1 Enfoque en técnicas

| Técnica | Aspecto a hallar | Columnas | Foco |
|---|---|---|---|
| **Jaccard + Shingling** | Similitud exacta entre descripciones en una muestra | `shingles` | P2 |
| **MinHash** | Firma compacta que aproxima Jaccard sin guardar los conjuntos completos | `shingles` | P2 |
| **LSH** (MinHashLSH) | Pares de procesos casi duplicados, y luego se filtra misma entidad + fechas cercanas | `shingles`, `entidad`, `fecha` | P2 |
| **ANN: IVF / HNSW** | Los *k* procesos más parecidos a uno dado (TF-IDF) para comparar sus montos | `tokens`, `monto_final` | P3 |
| **Bloom Filter** | ¿Este par entidad–proveedor ya apareció antes en el flujo? | `entidad`, `proveedor`, `fecha` | P2 |
| **Count-Min Sketch** | Proveedores y entidades más frecuentes (*heavy hitters*) | `proveedor`, `entidad` | P2 |
| **DGIM** | Cuántos procesos con postor único hubo en la ventana reciente del flujo | `postor_unico`, `fecha` | P1 |
| **Apriori / FP-Growth** | Reglas tipo `{CONTRATACIÓN DIRECTA, tramo alto} → {postor único}` | `canasta` | P1 |

### 4. Selección de columnas

Se queda sólo con las columnas útiles para los principales enfoques

In [11]:
# Mismo criterio de nulos que el EDA (sección 4.1): % de vacíos por columna
cols_simples = [c for c, t in base.dtypes if not t.startswith("array")]
nulos = (base.select([(100 * F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1)) / N)
         .alias(c) for c in cols_simples]).toPandas().T[0])

# Se descartan las columnas casi vacías y las que no aportan al análisis
descartar = sorted(set([c for c in cols_simples if nulos[c] > 95] + ["metodo", "codigo_proceso"]))
sel = base.drop(*descartar)

print("Descartadas:", descartar)
print("Seleccionadas:", sel.columns)

Descartadas: ['codigo_proceso', 'estado', 'metodo']
Seleccionadas: ['ocid', 'fecha', 'entidad', 'region', 'descripcion', 'items_desc', 'metodo_detalle', 'categoria', 'monto_referencial', 'moneda', 'n_postores', 'monto_adjudicado', 'proveedor', 'unspsc', 'anio']


Las columnas descartadas son las que se marcó con más de 95% de nulos (`estado`,`metodo`), más `codigo_proceso`, que no aportan mucho

### 5. Preprocesamiento y limpieza

Se procesa y limpia la data

#### 5.1 Auxiliares

In [12]:
# Texto de las categóricas en mayúsculas y sin espacios extra, para que agrupen bien
def normalizar_texto(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.upper(F.trim(F.col(c))))
    return df

# Si un ocid aparece más de una vez, nos quedamos con el registro más reciente
def quitar_duplicados(df):
    w = Window.partitionBy("ocid").orderBy(F.col("fecha").desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

# Sólo los años que cargamos (el EDA mostró algunas fechas de 2026)
def filtrar_anios(df, anios):
    return df.where(F.col("anio").isin(anios))

# Sólo soles; los procesos sin moneda se asumen en PEN
def filtrar_moneda_pen(df):
    return df.where(F.coalesce(F.col("moneda"), F.lit("PEN")) == "PEN").drop("moneda")

# Montos <= 0 pasan a nulo: no borramos la fila, solo el valor inválido
def limpiar_montos(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.when(F.col(c) > 0, F.col(c)))
    return df

# Sin descripción no hay similitud ni vecinos posibles
def filtrar_con_descripcion(df):
    return df.where(F.length(F.trim("descripcion")) > 0)

#### 5.2 Flujo

In [ ]:
# Aplicamos los filtros en orden y anotamos cuántas filas quedan en cada paso
pasos = [("staging", sel.count())]

limpio = normalizar_texto(sel, ["entidad", "region", "proveedor", "metodo_detalle", "categoria"])

limpio = quitar_duplicados(limpio)
pasos.append(("sin duplicados", limpio.count()))

limpio = filtrar_anios(limpio, anios)
pasos.append((f"fecha en {min(anios)}-{max(anios)}", limpio.count()))

limpio = filtrar_moneda_pen(limpio)
pasos.append(("moneda PEN", limpio.count()))

# Este paso no borra filas, solo anula los montos inválidos (por eso el conteo no baja)
limpio = limpiar_montos(limpio, ["monto_referencial", "monto_adjudicado"])
pasos.append(("montos <= 0 a nulo", limpio.count()))

limpio = filtrar_con_descripcion(limpio)
pasos.append(("con descripción", limpio.count()))

pd.DataFrame(pasos, columns=["paso", "filas"]).assign(perdidas=lambda d: d.filas.shift(1) - d.filas)

La tabla `pasos` muestra cuántas filas se pierden en cada filtro. Duplicados y años fuera de rango pierden pocas filas

#### 5.3 Cambios

- **Duplicados:** si un `ocid` aparece más de una vez se queda el registro más reciente
- **Montos ≤ 0:** se ponen en nulo, pero el proceso se conserva para texto y frecuencias
- **Outliers de monto:** no se eliminan. El EDA (4.2) mostró que los montos altos son obras reales, así que se agrupan en **tramos** en la sección 6
- **Fechas:** sólo los años cargados (2023–2025)
- **Texto** en mayúsculas y sin espacios extra
- **Moneda:** sólo PEN; los procesos sin moneda se asumen en PEN

### 6. Transformación

Se realizan las transformaciones debidas

#### 6.1 Auxiliares

In [14]:
con_tilde, sin_tilde = "áéíóúüñ", "aeiouun"

# Minúsculas y sin tildes, para que "licitación" y "licitacion" sean la misma palabra
def sin_tildes(c):
    return F.translate(F.lower(c), con_tilde, sin_tilde)

# texto = descripción + descripciones de los ítems, sin signos ni espacios dobles
def normalizar_texto_libre(df):
    return (df
            .withColumn("texto", F.concat_ws(" ", "descripcion", "items_desc"))
            .withColumn("texto", sin_tildes(F.col("texto")))
            .withColumn("texto", F.trim(F.regexp_replace(F.regexp_replace("texto", r"[^a-z0-9 ]", " "), r"\s+", " "))))

# Palabras de 3 letras a más, sin stopwords en español
def tokenizar(df):
    stop = [s.translate(str.maketrans(con_tilde, sin_tilde)) for s in StopWordsRemover.loadDefaultStopWords("spanish")]
    df = RegexTokenizer(inputCol="texto", outputCol="_tok", pattern=r"[^a-z]+", minTokenLength=3).transform(df)
    df = StopWordsRemover(inputCol="_tok", outputCol="_tok2", stopWords=stop).transform(df)
    # Se conservan las palabras repetidas: TF-IDF necesita la frecuencia, no solo la presencia
    return df.withColumnRenamed("_tok2", "tokens").drop("_tok")

# Trozos de k caracteres: el conjunto con el que se compara Jaccard / MinHash
def generar_shingles(df, k=5):
    return df.withColumn("shingles", F.when(F.length("texto") >= k,
            F.array_distinct(F.expr(f"transform(sequence(1, length(texto) - {k} + 1), i -> substring(texto, i, {k}))")))
            .otherwise(F.array("texto")))

# Columnas derivadas: monto usable, tramo, competencia, rubro y campos de tiempo
def agregar_tramos_monto(df):
    return (df
            .withColumn("monto_final", F.coalesce("monto_adjudicado", "monto_referencial"))
            .withColumn("tramo_monto",
                F.when(F.col("monto_final").isNull(), "0_sin_monto")
                 .when(F.col("monto_final") <    50_000, "1_<50k")
                 .when(F.col("monto_final") <   200_000, "2_50k-200k")
                 .when(F.col("monto_final") < 1_000_000, "3_200k-1M")
                 .when(F.col("monto_final") < 5_000_000, "4_1M-5M")
                 .otherwise("5_>5M"))
            .withColumn("postor_unico", (F.col("n_postores") == 1).cast("int"))   # nulo si no se reportó n_postores
            .withColumn("rubro", F.array_distinct(F.transform("unspsc", lambda x: F.substring(x, 1, 2))))
            .withColumn("mes", F.month("fecha"))
            .withColumn("periodo", F.date_format("fecha", "yyyy-MM")))

# Cada proceso como una transacción de ítems clave=valor
def construir_canasta(df):
    item = lambda k, c: F.when(F.col(c).isNotNull(), F.concat(F.lit(f"{k}="), F.col(c).cast("string")))
    return df.withColumn("canasta", F.array_distinct(F.concat(
            F.array_compact(F.array(
                item("cat", "categoria"), item("met", "metodo_detalle"), item("reg", "region"),
                item("monto", "tramo_monto"),
                F.when(F.col("postor_unico") == 1, F.lit("comp=postor_unico"))
                 .when(F.col("n_postores") > 1, F.lit("comp=varios_postores")))),
            F.transform("rubro", lambda r: F.concat(F.lit("rubro="), r)))))

#### 6.2 Flujo

In [ ]:
# Primero el texto: normalizar, tokenizar y armar los shingles
t = normalizar_texto_libre(limpio)
t = tokenizar(t)
t = generar_shingles(t)
t.select("texto", "tokens", F.slice("shingles", 1, 6).alias("shingles (6 primeros)")).show(3, truncate=60)

# Luego las columnas derivadas y la canasta
t = agregar_tramos_monto(t)
t = construir_canasta(t)

# Comprobación: a menor tramo de monto, menor competencia
t.groupBy("tramo_monto").agg(F.count("*").alias("procesos"), F.round(100 * F.avg("postor_unico"), 1).alias("%_postor_unico")).orderBy("tramo_monto").show()
t.select("canasta").show(3, truncate=110)

`tokens` y `shingles` quedan limpios y comparables para Jaccard/MinHash/LSH y TF-IDF. El % de postor único
por tramo confirma montos bajos, menor competencia

#### 6.3 Cambios

- **`monto_final`**, **`tramo_monto`**, **`postor_unico`**, **`rubro`** (segmento UNSPSC de 2 dígitos)

- **`texto`** = `descripcion` + descripciones de los ítems en minúsculas, sin tildes ni signos. La similitud se calcula sobre este texto, no solo sobre `descripcion`

- **`canasta`**: cada proceso como transacción `clave=valor` (para Apriori / FP-Growth)

- **`shingles`**: k-shingles de 5 caracteres, sin repetir (Jaccard / MinHash / LSH trabajan con conjuntos)

- **`tokens`**: palabras sin *stopwords* en español, **con repeticiones**, porque TF-IDF necesita la frecuencia de cada palabra

- **`monto_final`** mezcla precio final y estimado (usa `monto_adjudicado` y, si falta, `monto_referencial`). Para comparar precios (P3) conviene mirar también `monto_adjudicado` y `monto_referencial`, que se guardan aparte

- **`postor_unico`** queda nulo cuando el proceso no reportó `n_postores`; esos casos se excluyen del flujo de DGIM, no cuentan como cero

### 7. Columnas por técnica

El dataset ya tiene todo lo necesario. Ahora se elige, técnica por técnica, con qué columnas se va a trabajar.

#### 7.1 Búsqueda por similitud (Jaccard / MinHash / LSH)

`shingles` es el conjunto que se compara; `entidad` y `fecha` sirven para quedarse solo con los pares
de la misma entidad y en fechas cercanas (P2).

In [ ]:
# Conjunto de shingles por proceso, más los campos para filtrar los pares
cols_similitud = ["ocid", "entidad", "fecha", "texto", "shingles"]
sim = t.select(*cols_similitud)

sim.limit(3).select("ocid", "entidad", F.size("shingles").alias("n_shingles")).show(truncate=40)

#### 7.2 Vecinos más cercanos (TF-IDF / IVF / HNSW)

`tokens` se convierte en el vector TF-IDF y los montos son lo que se compara entre procesos parecidos (P3).

In [ ]:
# Tokens para vectorizar y montos para comparar precios
cols_vecinos = ["ocid", "descripcion", "tokens", "region", "monto_referencial", "monto_adjudicado", "monto_final"]
vec = t.select(*cols_vecinos)

vec.limit(3).select("ocid", F.size("tokens").alias("n_tokens"), "monto_final").show(truncate=40)

#### 7.3 Streaming (Bloom / Count-Min / DGIM)

El flujo se recorre en orden de `fecha`: `entidad` y `proveedor` alimentan Bloom y Count-Min,
y `postor_unico` es el bit que cuenta DGIM (P1, P2).

In [ ]:
# El orden por fecha es lo que convierte la tabla en un flujo
cols_streaming = ["ocid", "fecha", "entidad", "proveedor", "n_postores", "postor_unico"]
stream = t.select(*cols_streaming).orderBy("fecha")

stream.limit(3).show(truncate=40)

#### 7.4 Reglas de asociación (Apriori / FP-Growth)

`canasta` es la transacción lista para minar; se guardan también las columnas con las que se armó,
por si hay que rehacerla con otros tramos (P1).

In [ ]:
# Transacción por proceso y los ítems que la componen
cols_reglas = ["ocid", "categoria", "metodo_detalle", "region", "tramo_monto", "rubro", "unspsc", "canasta"]
reglas = t.select(*cols_reglas)

reglas.limit(3).select("ocid", "canasta").show(truncate=100)

#### 7.5 Recuento

In [ ]:
# Resumen de qué columnas usa cada técnica
recuento = pd.DataFrame([
    ("Similitud (Jaccard / MinHash / LSH)",  ", ".join(cols_similitud)),
    ("Vecinos (TF-IDF / IVF / HNSW)",        ", ".join(cols_vecinos)),
    ("Streaming (Bloom / Count-Min / DGIM)", ", ".join(cols_streaming)),
    ("Reglas (Apriori / FP-Growth)",         ", ".join(cols_reglas)),
], columns=["técnica", "columnas"])

with pd.option_context("display.max_colwidth", 200):
    display(recuento)

- **Similitud** → `shingles` (+ `entidad`, `fecha` para filtrar los pares)
- **Vecinos** → `tokens` (+ `monto_final`, `monto_referencial`, `monto_adjudicado`, `region`)
- **Streaming** → `fecha` (orden), `entidad`, `proveedor`, `postor_unico`
- **Reglas** → `canasta` (armada con `categoria`, `metodo_detalle`, `region`, `tramo_monto`, `rubro`)

### 8. Dataset final

Se guarda en Parquet particionado por `anio` y `mes`: los proximos pasos leen solo lo que necesitan, y si
filtran por año o mes, Spark salta carpetas enteras sin abrirlas (*partition pruning*)

#### 8.1 Auxiliares

In [16]:
# Guarda el dataset partido en carpetas anio=/mes=
def guardar_particionado(df, columnas, path):
    final = df.select(*columnas)
    final.write.mode("overwrite").partitionBy("anio", "mes").parquet(str(path))

# Relee lo guardado y confirma esquema, filas y particiones
def verificar_dataset(path):
    procesos = spark.read.parquet(str(path))
    procesos.printSchema()
    print(f"Filas finales: {procesos.count():,} | particiones en disco: "
          f"{len(list(path.glob('anio=*/mes=*')))} carpetas anio=/mes=")
    procesos.groupBy("anio").count().orderBy("anio").show()
    return procesos

#### 8.2 Flujo

In [ ]:
# Columnas finales: la unión de las cuatro listas de la sección 7, más los campos de tiempo
columnas = ["ocid", "fecha", "anio", "mes", "periodo",
    "entidad", "region", "proveedor",
    "categoria", "metodo_detalle", "n_postores", "postor_unico",
    "monto_referencial", "monto_adjudicado", "monto_final", "tramo_monto",
    "descripcion", "texto", "tokens", "shingles",
    "unspsc", "rubro", "canasta"]

# Aviso si alguna técnica pide una columna que no se está guardando
pedidas = dict.fromkeys(cols_similitud + cols_vecinos + cols_streaming + cols_reglas)
print("Faltantes:", [c for c in pedidas if c not in columnas])

guardar_particionado(t, columnas, parquets / "procesos")
procesos = verificar_dataset(parquets / "procesos")

# El plan físico debe mostrar PartitionFilters para anio/mes
procesos.where("anio = 2025 AND mes = 3").select("ocid").explain()

El plan físico debe mostrar PartitionFilters para anio/mes

### 9. Overview

#### 9.1 Próximos pasos

La data cumple los cuatro criterios necesarios (EDA 8.2) y queda limpia en `parquet/procesos/`.
El detalle de columnas por técnica está en el recuento de 7.5:

| Método | Columnas |
|---|---|
| Jaccard / MinHash / LSH | `shingles` (+ `entidad`, `fecha` para filtrar pares) |
| ANN (IVF / HNSW) | `tokens` → TF-IDF, `monto_final` |
| Bloom / Count-Min / DGIM | `fecha` (orden), `entidad`, `proveedor`, `postor_unico` |
| Apriori / FP-Growth | `canasta` |

#### 9.2 Uso de la data

```python
procesos = spark.read.parquet("parquet/procesos")
procesos_2025 = spark.read.parquet("parquet/procesos").where("anio = 2025")
```

### 10. Cerrar sesión de Spark

Libera los recursos del cluster cerrando la SparkSession.

In [18]:
# Liberamos memoria y executors del cluster
spark.stop()